In [1]:
!pip install streamlit pyngrok



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 23.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 10.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 59.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.0/83.0 kB 5.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.7/62.7 kB 5.5 MB/s eta 0:00:00


In [2]:
!ngrok authtoken 2idetfHrlP3Ss2iCYGXVgnDdvZt_5Kq18oRHqivx7GnEHn1Yn


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [9]:
%%writefile app.py
import streamlit as st
import os
import random
import tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from sklearn.metrics import classification_report
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# Setting up Streamlit title and description
st.title('MediScan: AI-Powered Medical Image Analysis for Disease Diagnosis')
st.write("This application helps in diagnosing eye diseases using AI.")

# File Upload for Kaggle API
st.sidebar.title("Upload Kaggle.json File")
uploaded_file = st.sidebar.file_uploader("Upload your kaggle.json file", type="json")

if uploaded_file:
    with open('kaggle.json', 'wb') as f:
        f.write(uploaded_file.read())
    st.sidebar.success('Uploaded kaggle.json')

    # Configure Kaggle API
    os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
    os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
    os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)

    # Download dataset using Kaggle CLI
    st.sidebar.write("Downloading dataset...")
    os.system('kaggle datasets download -d gunavenkatdoddi/eye-diseases-classification')
    os.system('unzip -q eye-diseases-classification.zip -d dataset')

    st.sidebar.success('Dataset downloaded and extracted')

# Preprocess Images
st.header('Preprocessing Images')
rescale = tf.keras.layers.Rescaling(1./255)

# Load train and validation datasets
if os.path.exists('dataset'):
    train_ds = tf.keras.utils.image_dataset_from_directory(
        directory='dataset/dataset',
        batch_size=32,
        image_size=(256, 256),
        validation_split=0.2,
        subset="training",
        seed=123,
        label_mode='categorical',  # Ensure labels are one-hot encoded
    )
    train_ds = train_ds.map(lambda x, y: (rescale(x), y))

    validation_ds = tf.keras.utils.image_dataset_from_directory(
        directory='dataset/dataset',
        batch_size=32,
        image_size=(256, 256),
        validation_split=0.2,
        subset="validation",
        seed=123,
        label_mode='categorical',  # Ensure labels are one-hot encoded
    )
    validation_ds = validation_ds.map(lambda x, y: (rescale(x), y))

    test_ds = tf.keras.utils.image_dataset_from_directory(
        directory='dataset/dataset',
        batch_size=32,
        image_size=(256, 256),
        label_mode='categorical',
        shuffle=False,
    )
    test_ds = test_ds.map(lambda x, y: (rescale(x), y))

    st.success('Image datasets loaded successfully')
else:
    st.error('Dataset directory not found. Please upload the kaggle.json file and download the dataset.')

# Visualize Sample Images
@st.cache
def visualize_images(path, target_size=(256, 256), num_images=5):
    image_filenames = [f for f in os.listdir(path) if os.path.isfile(os.path.join(path, f))]
    if not image_filenames:
        raise ValueError("No images found in the specified path")

    selected_images = random.sample(image_filenames, min(num_images, len(image_filenames)))
    fig, axes = plt.subplots(1, num_images, figsize=(15, 3), facecolor='white')

    for i, image_filename in enumerate(selected_images):
        image_path = os.path.join(path, image_filename)
        image = Image.open(image_path)
        image = image.resize(target_size)
        axes[i].imshow(image)
        axes[i].axis('off')
        axes[i].set_title(image_filename)

    plt.tight_layout()
    st.pyplot(fig)

if os.path.exists('dataset/dataset'):
    st.header('Sample Images')
    path_to_visualize = st.selectbox('Select Category', ['dataset/dataset/cataract', 'dataset/dataset/glaucoma', 'dataset/dataset/diabetic_retinopathy', 'dataset/dataset/normal'])
    visualize_images(path_to_visualize, num_images=5)
else:
    st.error('Dataset directory not found. Please upload the kaggle.json file and download the dataset.')

# Build and Train the Model
st.header('Model Training')

# Define the model
model = tf.keras.Sequential([
    tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Conv2D(64, (3, 3), activation='relu'),
    tf.keras.layers.MaxPooling2D((2, 2)),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(4, activation='softmax')  # Output layer with 4 units for 4 classes
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

st.write(model.summary())

# Train the model
if st.button('Train Model'):
    if os.path.exists('dataset/dataset'):
        train_ds = train_ds.prefetch(buffer_size=tf.data.AUTOTUNE)
        validation_ds = validation_ds.prefetch(buffer_size=tf.data.AUTOTUNE)

        with st.spinner('Training...'):
            history = model.fit(
                train_ds,
                validation_data=validation_ds,
                epochs=5,
            )
        st.success('Training completed')
        st.write(history.history)
    else:
        st.error('Dataset directory not found. Please upload the kaggle.json file and download the dataset.')


# Test the Model and Display Results
st.header('Model Evaluation')

# Evaluate the model
if st.button('Evaluate Model'):
    if os.path.exists('dataset/dataset'):
        with st.spinner('Evaluating...'):
            test_loss, test_acc = model.evaluate(test_ds)
        st.write(f'Test Accuracy: {test_acc}')
        st.write(f'Test Loss: {test_loss}')

        # Generate classification report
        y_pred = np.argmax(model.predict(test_ds), axis=-1)
        y_true = np.concatenate([y for x, y in test_ds], axis=0)
        y_true = np.argmax(y_true, axis=1)
        report = classification_report(y_true, y_pred, target_names=['cataract', 'diabetic_retinopathy', 'glaucoma', 'normal'])
        st.text(report)
    else:
        st.error('Dataset directory not found. Please upload the kaggle.json file and download the dataset.')


Overwriting app.py


In [10]:
from pyngrok import ngrok

ngrok.kill()


In [11]:
public_url = ngrok.connect(8501)
print(public_url)


NgrokTunnel: "https://fa37-34-68-200-152.ngrok-free.app" -> "http://localhost:8501"


In [12]:
!streamlit run app.py &>/dev/null&
